<a href="https://colab.research.google.com/github/kuberiitb/shopping_agent/blob/main/notebooks/02_instamart_items_scraping_with_skuId.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import pickle
import random
import time
import pandas as pd
import requests
import json
from bs4 import BeautifulSoup
from tqdm import notebook

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Expires": "0",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

In [128]:
item_base_url = "https://www.swiggy.com/instamart/item/"
urls = list(set([
    "https://www.swiggy.com/instamart/category-listing?categoryName=Meat%20and%20Seafood&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Atta,%20Rice%20and%20Dal&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Masalas&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Oils%20and%20Ghee&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cereals%20and%20Breakfast&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cereals%20and%20Breakfast&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%205&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Ice%20Creams%20and%20Frozen%20Desserts&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Chocolates&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Biscuits%20and%20Cakes&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Tea,%20Coffee%20and%20Milk%20drinks&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Sauces%20and%20Spreads&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Noodles,%20Pasta,%20Vermicelli&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Cleaners%20and%20Repellents&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Bath%20and%20Body&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Makeup&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Feminine%20Hygiene&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Baby%20Care&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Health%20and%20Pharma&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Skincare&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fashion&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Electronics%20and%20Appliances&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fresh%20Fruits&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/category-listing?categoryName=Fresh%20Vegetables&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false",
    "https://www.swiggy.com/instamart/collection-listing?collectionId=72456&custom_back=true",
]))


In [3]:
def get_data(url):
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "html.parser")
  scripts = soup.find_all("script")

  raw_json = None

  for script in scripts:
      if script.string and "ItemCollectionCard" in script.string:
          raw_json = extract_first_json(
              script.string,
              '{"@type":"type.googleapis.com/swiggy.im.v1.ItemCollectionCard"'
          )
          break

  data = json.loads(raw_json)['items']
  return data


In [4]:
def extract_first_json(text, start_key):
    start = text.find(start_key)
    if start == -1:
        return None

    brace_count = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        char = text[i]

        if char == '"' and not escape:
            in_string = not in_string

        if not in_string:
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1

        if char == '\\' and not escape:
            escape = True
        else:
            escape = False

        if brace_count == 0:
            return text[start:i+1]

    return None

def extract_quantity(title):
    match = re.search(r'(\d+\s?(g|gm|kg|ml|l))', title.lower())
    return match.group(0) if match else None

def extract_items_from_dict(d, keys):
  out = {}
  for key in keys:
    if type(key)==str:
      out[key] = d[key]
    elif type(key)==dict:
      # print("dict key", key)

      for mainkey, subkeys in key.items():
        out[mainkey] = {}
        # print("mainkey", mainkey)
        # print("subkeys", subkeys)
        temp = {}
        for mainkeydata in d[mainkey]:
          for subkey in subkeys:
            temp[subkey] = mainkeydata[subkey]
          out[mainkey] = temp
  return out



In [5]:
#base image url https://instamart-media-assets.swiggy.com/swiggy/image/upload/fl_lossy,f_auto,q_auto,h_600/

In [7]:
# main_data['3K4IJLW5DH']

# {'displayName': 'Onion (Eerulli)',
#  'brand': 'Fruits and Vegetables',
#  'parentProductId': 'TXX0TEK4FU',
#  'category': 'Vegetables',
#  'imageIds': 'NI_CATALOG/IMAGES/CIW/2026/3/24/a571cc74-b070-43d5-8578-d414312f0551_2116_1.jpg',
#  'skuId': '3K4IJLW5DH',
#  'quantityDescription': '1 kg',
#  'currencyCode': 'INR',
#  'mrp': '36',
#  'offerPrice': '29'}

## Appendix: Can we get skuId and extract details from respective page?

In [8]:
# item page pattern
# https://www.swiggy.com/instamart/item/MQBV46M8S1

## extract all SKUs from a category page

In [166]:
# testing
url = urls[1]
# "https://www.swiggy.com/instamart/category-listing?categoryName=Tea,%20Coffee%20and%20Milk%20drinks&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
sku_ids = []

scripts = soup.find_all("script")

for script in scripts:
    content = script.string or script.get_text()
    if not content:
        continue

    matches = re.findall(r'"productId"\s*:\s*"([^"]+)"', content)
    for match in matches:
        sku_ids.append(match)

print(sku_ids)

[]


In [134]:
def extract_skus(url):
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "html.parser")
  sku_ids = []

  scripts = soup.find_all("script")

  for script in scripts:
      content = script.string or script.get_text()
      if not content:
          continue

      matches = re.findall(r'"productId"\s*:\s*"([^"]+)"', content)
      for match in matches:
          sku_ids.append(match)

  return sku_ids

In [119]:
try:
  #load existing data(if available)
  main_data = pickle.load(open('main_data.pkl','rb'))
  print(len(main_data))
except:
  main_data = {}

341


In [170]:
# sometimes swiggy blocks this extraction
# save such unprocessed urls for next iteration
blank_urls = []

In [152]:
for url in urls:
  print(url)
  sku_ids = extract_skus(url)
  print(sku_ids)
  if sku_ids==[]:
    blank_urls.append(url)
    continue

  for sku_id in sku_ids:
    item_url = item_base_url + sku_id
    print(item_url)
    try:
      del data
    except:
      pass
    try:
      data = get_data(item_url)
    except Exception as e:
      # print(e)
      continue

    extracted_data = []
    for item in data:
      print(item)
      out = extract_items_from_dict(item, ['displayName','brand', 'parentProductId', {'variations':['category', 'price', 'imageIds', 'skuId','quantityDescription']}])
      extracted_data.append(out)

    df = pd.DataFrame(extracted_data)
    #convert hierarchical columns to flat
    new_cols = pd.json_normalize(df['variations'])

    df = df.drop('variations',axis=1).join(new_cols)
    df['imageIds'] = df['imageIds'].apply(lambda x:x[0] if type(x)==list else x)
    df = df.loc[:, [x for x in df.columns if ('price' not in x) or (x in ['price.mrp.currencyCode','price.mrp.units','price.offerPrice.units'])] ]
    df = df.rename(columns={'price.mrp.currencyCode':'currencyCode',
                      'price.mrp.units':'mrp',
                      'price.offerPrice.units':'offerPrice'
    })

    # print(df)

    #updating records if there is new information(some data has mrp information missing)
    for _, k in df.iterrows():
      data_dict = k.to_dict()
      if (data_dict['skuId'] not in main_data) or ((data_dict['skuId'] in main_data) and ('mrp' not in main_data.get(data_dict['skuId'])) ):
        main_data[data_dict['skuId']] = data_dict

  print("Items count", len(main_data))
  print("Items with images", len([x for _, x in main_data.items() if x.get('imageIds')]))
  print("Items with MRP", len([x for _, x in main_data.items() if x.get('mrp')]))

  pickle.dump(main_data, open('main_data.pkl','wb'))
  print("Dump updated")


https://www.swiggy.com/instamart/category-listing?categoryName=Noodles,%20Pasta,%20Vermicelli&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false
[]
https://www.swiggy.com/instamart/category-listing?categoryName=Health%20and%20Pharma&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2014&showAgeConsent=false
[]
https://www.swiggy.com/instamart/category-listing?categoryName=Electronics%20and%20Appliances&storeId=911033&offset=0&filterName=&taxonomyType=IM%20Meatsy&showAgeConsent=false
[]
https://www.swiggy.com/instamart/category-listing?categoryName=Sauces%20and%20Spreads&storeId=911033&offset=0&filterName=&taxonomyType=taxonomy%2010&showAgeConsent=false
[]
https://www.swiggy.com/instamart/category-listing?categoryName=Fresh%20Vegetables&storeId=911033&offset=0&filterName=&taxonomyType=Speciality%20taxonomy%201&showAgeConsent=false
[]
https://www.swiggy.com/instamart/category-listing?categoryName=Feminine%20Hygiene&storeId=911033&offset=0&filterNa

In [167]:

print("Items count", len(main_data))
print("Items with images", len([x for _, x in main_data.items() if x.get('imageIds')]))
print("Items with MRP", len([x for _, x in main_data.items() if x.get('mrp')]))

pickle.dump(main_data, open('main_data.pkl','wb'))
print("Dump updated")


Items count 332
Items with images 332
Items with MRP 332
Dump updated


In [168]:
df = pd.DataFrame(main_data).T
df.to_csv("instamart_items.csv",index=False)

In [169]:
df.shape

(332, 10)